In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.metrics import r2_score , root_mean_squared_error , mean_absolute_error
from sklearn.compose import TransformedTargetRegressor
import optuna
from lightgbm import LGBMRegressor
import mlflow
import dagshub
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate , plot_slice 

In [16]:
dagshub.init(repo_owner='mridul0010', repo_name='NYC-Taxi-Trip-Duration', mlflow=True)

Initialized MLflow to track repo "mridul0010/NYC-Taxi-Trip-Duration"

Repository mridul0010/NYC-Taxi-Trip-Duration initialized!

In [17]:
mlflow.set_tracking_uri("https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow")

In [18]:
mlflow.set_experiment("1. HyperParameter Tuning - LGBM")

<Experiment: artifact_location='mlflow-artifacts:/72b7e5917a034f9c8dbded2ff121b240', creation_time=1784487187233, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1784487187233, lifecycle_stage='active', name='1. HyperParameter Tuning - LGBM', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [ ]:
X_train = pd.read_csv("../data/processed/baseline/features.csv")
X_test  = pd.read_csv("../data/processed/baseline/features_test.csv")
y_train = pd.read_csv("../data/processed/baseline/labels.csv")
y_test = pd.read_csv("../data/processed/baseline/labels_test.csv")

In [20]:
print("Shape of X_train :-",X_train.shape)
print("Shape of y_train :-",y_train.shape)
print("Shape of X_test :-",X_test.shape)
print("Shape of y_test :-",y_test.shape)

Shape of X_train :- (1125732, 28)
Shape of y_train :- (1125732, 1)
Shape of X_test :- (281434, 28)
Shape of y_test :- (281434, 1)


In [21]:
pd.set_option('display.max_columns' , None)

In [22]:
y_train = y_train.values.ravel()

In [23]:
def objective(trial):
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model", "LightGBM")
        mlflow.set_tag("model_type", "Regressor")
        
        param = {
            "n_estimators": trial.suggest_int("n_estimators", 400, 1000), 
            "max_depth": trial.suggest_int("max_depth", 5, 12),
            "num_leaves": trial.suggest_int("num_leaves", 31, 256),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 20, 100),
            "min_gain_to_split": trial.suggest_float("min_gain_to_split", 1e-3, 5.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1, 20),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),            
            "subsample": trial.suggest_float("subsample", 0.6, 0.95),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.85),
            "random_state": 42
        }
        
        lgbm = LGBMRegressor(**param)
        model = TransformedTargetRegressor(
            regressor= lgbm,
            func=np.log1p,
            inverse_func=np.expm1
        )

        
        model.fit(X_train, y_train)
        
        cv_score = cross_val_score(
            model ,
            X_train, 
            y_train , 
            cv=10 , 
            scoring="neg_mean_absolute_error",
            n_jobs=1
        )

        mean_score = -(cv_score.mean())
        
        mlflow.log_params(param)
        mlflow.log_metric("CV Score", mean_score)
        
        return mean_score

In [24]:
study = optuna.create_study(direction='minimize')

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective , n_trials=10 , n_jobs=1 , show_progress_bar=True)

    # log the best params
    mlflow.log_params(study.best_params)

    # log best score
    mlflow.log_metric("best_score" , study.best_value)

    # training the lgbm on best param
    best_lgbm = LGBMRegressor(**study.best_params , n_jobs = -1)

    best_model = TransformedTargetRegressor(
        regressor=best_lgbm,
        func=np.log1p,
        inverse_func=np.expm1
    )

    best_model.fit(X_train , y_train)

    y_pred_train = best_model.predict(X_train)
    y_pred_test = best_model.predict(X_test)

    scores = cross_val_score(
        best_model,
        X_train,
        y_train,
        scoring="neg_mean_absolute_error",
        cv=5,
        n_jobs=1
    )

    # logging metrics
    mlflow.log_metric("Training_error_MAE" ,mean_absolute_error(y_train ,y_pred_train))
    mlflow.log_metric("Test_error_MAE" ,mean_absolute_error(y_test ,y_pred_test))
    mlflow.log_metric("Training_error_RMSE" ,root_mean_squared_error(y_train ,y_pred_train))
    mlflow.log_metric("Test_error_RMSE" ,root_mean_squared_error(y_test ,y_pred_test))
    mlflow.log_metric("Training_r2" ,r2_score(y_train ,y_pred_train))
    mlflow.log_metric("Test_r2" ,r2_score(y_test ,y_pred_test))
    mlflow.log_metric("cross_val" , -scores.mean())

    # Generate the optuna plots
    fig_history = plot_optimization_history(study)
    fig_parallel = plot_parallel_coordinate(study)
    fig_importance = plot_param_importances(study)
    fig_slice = plot_slice(study)

    # Loginf plots
    mlflow.log_figure(fig_history, "optuna_plots/optimization_history.html")
    mlflow.log_figure(fig_importance, "optuna_plots/param_importances.html")
    mlflow.log_figure(fig_parallel, "optuna_plots/parallel_coordinate.html")    
    mlflow.log_figure(fig_slice, "optuna_plots/plot_slice.html")

    # log the best model 
    mlflow.sklearn.log_model(
        sk_model=best_model, 
        name="model_LGBM",
        serialization_format="cloudpickle"
    )

[I 2026-07-20 00:24:03,272] A new study created in memory with name: no-name-41011413-bf74-4b93-b395-0fcbb2d94a6d


  0%|          | 0/10 [00:00<?, ?it/s]

[LightGBM] [Warning] min_gain_to_split is set=0.24923926003008923, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.24923926003008923
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_gain_to_split is set=0.24923926003008923, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.24923926003008923
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.028828 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1212
[LightGBM] [Info] Number of data points in the train set: 1125732, number of used features: 27
[LightGBM] [Info] Start training from score 2.519122
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positi

2026/07/20 00:59:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/6/runs/c19fa750b3d74c44bed637f4b3db6cb6
🧪 View experiment at: https://dagshub.com/mridul0010/NYC-Taxi-Trip-Duration.mlflow/#/experiments/6


In [25]:
study.best_value

3.0609443236529863

In [26]:
study.best_params

{'n_estimators': 849,
 'max_depth': 11,
 'num_leaves': 168,
 'learning_rate': 0.038346126061416304,
 'min_child_samples': 58,
 'min_gain_to_split': 0.003987966178171963,
 'reg_lambda': 3.642945821840642,
 'reg_alpha': 0.0032468327743102243,
 'subsample': 0.8508372416733088,
 'colsample_bytree': 0.8488806473646566}

In [27]:
best_lgbm = LGBMRegressor(**study.best_params)

model = TransformedTargetRegressor(
        regressor=best_lgbm,
        func=np.log1p,
        inverse_func=np.expm1
    )

model.fit(X_train , y_train)

[LightGBM] [Warning] min_gain_to_split is set=0.003987966178171963, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.003987966178171963
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_gain_to_split is set=0.003987966178171963, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.003987966178171963
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.055512 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1209
[LightGBM] [Info] Number of data points in the train set: 1125732, number of used features: 27
[LightGBM] [Info] Start training from score 2.519122
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",LGBMRegressor...8372416733088)
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[<U38](28,)","['target_encoded__pickup_zone','target_encoded__dropoff_zone', 'target_encoded__route_time_density',..., 'standard_scaled__is_interstate_trip','standard_scaled__is_late_night', 'standard_scaled__is_weekend_night']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying regressor exposes such an attribute when fit... versionadded:: 0.24,int,28
regressor_ regressor_: objectFitted regressor.,LGBMRegressor,LGBMRegressor...8372416733088)
transformer_ transformer_: objectTransformer used in :meth:`fit` and :meth:`predict`.,FunctionTransformer,FunctionTrans...validate=True)
,num_leaves,168


In [34]:
y_pred = model.predict(X_test)

r2score = r2_score(y_test , y_pred)
rmse = root_mean_squared_error(y_test , y_pred)
mae = mean_absolute_error(y_test , y_pred)

print("R2 Score :-",r2score)
print("RMSE :-",rmse)
print("MAE :-",mae)

[LightGBM] [Warning] min_gain_to_split is set=0.003987966178171963, min_split_gain=0.0 will be ignored. Current value: min_gain_to_split=0.003987966178171963
R2 Score :- 0.790974777422908
RMSE :- 4.982971408652478
MAE :- 3.063583873654746


In [29]:
fig_history = plot_optimization_history(study)
fig_parallel = plot_parallel_coordinate(study)
fig_importance = plot_param_importances(study)
fig_slice = plot_slice(study)

In [30]:
fig_history.show()

In [31]:
fig_parallel.show()

In [32]:
fig_importance.show()

In [33]:
fig_slice.show()